# Generative AI Real Estate Analysis with Gemini

This notebook extends the real estate project by integrating Google's Gemini API with the property dataset. It prepares a manageable sample of active listings, prompts Gemini to identify market patterns, asks the model to estimate property prices from listing characteristics, and compares the generated estimates with actual listing prices.

> **Security note:** The Gemini API key is loaded from the `GEMINI_API_KEY` environment variable and is never stored in this notebook.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

This cell imports several Python libraries that will be used throughout the notebook for data manipulation, visualization, and machine learning tasks:
*   `numpy` (as `np`): A fundamental package for numerical computation, especially with arrays.
*   `matplotlib.pyplot` (as `plt`): A plotting library for creating static, interactive, and animated visualizations.
*   `seaborn` (as `sns`): A data visualization library based on matplotlib, providing a high-level interface for drawing attractive and informative statistical graphics.
*   `sklearn.model_selection.train_test_split`: A function to split arrays or matrices into random train and test subsets.
*   `sklearn.linear_model.LinearRegression`: A machine learning model for performing linear regression.
*   `sklearn.metrics.mean_squared_error`: A function to calculate the mean squared error, a common metric for regression models.
*   `sklearn.metrics.r2_score`: A function to calculate the R-squared (coefficient of determination), another common metric for regression models.

In [ ]:
import os
import google.generativeai as genai

# Read the API key from an environment variable instead of storing it in the notebook
api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise ValueError("Set the GEMINI_API_KEY environment variable before running this notebook.")

genai.configure(api_key=api_key)


### Configure Gemini securely

The Gemini client is configured using an environment variable so credentials are not exposed in the public repository. Before running the notebook, set `GEMINI_API_KEY` in your local environment or notebook runtime.


You are using models that cost money!  However, Google allows for some free usage of their models per day.  So, if you get blocked because you reached your limit (rate limitted), simply change the model to some other model.  Below, I have included the code that lists all the availble models, and here is link to the rate limits:  https://ai.google.dev/gemini-api/docs/rate-limits.
Older models allow for more free access than newer models.  Stick to base Gemini models for text generation.

In [ ]:
for model in genai.list_models():
  print(model.name)

This cell lists all the available Generative AI models that you can access through the Gemini API configured with your API key.
*   `for model in genai.list_models():`: This loop iterates through a list of all models returned by `genai.list_models()`.
*   `print(model.name)`: For each model, it prints its name, which you can then use to specify a particular model (e.g., `'gemini-pro-latest'`) for generating content.

In [ ]:
## you can use the lastest model, but you get only so many free tokens
## flash light is an older model, and allows more requests
## see: https://ai.google.dev/gemini-api/docs/rate-limits
model = genai.GenerativeModel('gemini-pro-latest')

In [ ]:
prompt = "What is the capital of France?"
response = model.generate_content(prompt)
print(response.text)

This cell demonstrates how to send a simple text prompt to the initialized Gemini model and print its response.
*   `prompt = "What is the capital of France?"`: Defines a string variable `prompt` containing the question to be sent to the model.
*   `response = model.generate_content(prompt)`: Calls the `generate_content` method of the `model` object, passing the `prompt` to it. The model processes the prompt and returns a `response` object.
*   `print(response.text)`: Extracts the generated text content from the `response` object and prints it to the console.

In [ ]:
import pandas as pd

# Load the real estate dataset from the project's data folder
df = pd.read_csv("../data/realtor-data.zip.csv")

print("Dataset loaded successfully.")
display(df.head())


### Load the real estate dataset

The dataset is loaded from the repository's `data` folder using a relative path. This keeps the notebook portable and avoids dependencies on a personal Google Drive location.


In [ ]:
# Keep active listings for the generative AI analysis
df_cleaned = df[df['status'] == 'for_sale'].copy()

# Convert the cleaned data to CSV text for use in prompts
cleaned_csv_string = df_cleaned.to_csv(index=False)

print("Original number of rows:", len(df))
print("Number of active listings:", len(df_cleaned))
display(df_cleaned.head())


### Prepare active listings

Only records with `status == 'for_sale'` are retained for the Gemini analysis. The filtered DataFrame is then converted to CSV text so a small sample can be embedded directly in prompts.


In [ ]:
print("Characters in CSV:", len(cleaned_csv_string))
print("Number of rows:", len(cleaned_csv_string.splitlines()) - 1)

This cell provides a quick check of the size and number of rows in the `cleaned_csv_string`.
*   `print("Characters in CSV:", len(cleaned_csv_string))`: Prints the total number of characters in the `cleaned_csv_string`, giving an idea of its overall size.
*   `print("Number of rows:", len(cleaned_csv_string.splitlines()) - 1)`: Splits the `cleaned_csv_string` into individual lines (each representing a row) and counts them. It subtracts 1 to exclude the header row, providing the actual number of data rows.

In [ ]:
# Split the CSV into lines
csv_lines = cleaned_csv_string.splitlines()

# Keep the header and the first 200 data rows
header = csv_lines[0]
sample_rows = csv_lines[1:201]

# Rebuild a smaller CSV string
sample_csv = "\n".join([header] + sample_rows)

print("Sample characters:", len(sample_csv))
print("Sample rows:", len(sample_rows))

This cell creates a smaller sample of the cleaned real estate data for efficient processing by the language model, as very large inputs can be slow or hit token limits.
*   `csv_lines = cleaned_csv_string.splitlines()`: Splits the `cleaned_csv_string` into a list of strings, where each string represents a line (row) from the CSV.
*   `header = csv_lines[0]`: Extracts the first line from the `csv_lines` list, which is the CSV header.
*   `sample_rows = csv_lines[1:201]`: Selects the first 200 data rows (from index 1 to 200) from the `csv_lines` list.
*   `sample_csv = "\\n".join([header] + sample_rows)`: Reconstructs a new CSV string by joining the `header` and the `sample_rows` back together with newline characters, forming `sample_csv`.
*   `print("Sample characters:", len(sample_csv))`: Prints the total number of characters in the newly created `sample_csv` string.
*   `print("Sample rows:", len(sample_rows))`: Prints the number of data rows included in the `sample_csv`.

In [ ]:

prompt = f"""
Analyze the following sample of real estate data provided in CSV format:
{sample_csv}
Based on this sample, provide insights into:
1. Price trends
2. Bedrooms and bathrooms
3. House size and acreage
4. Geographic patterns
5. Missing or unusual values
6. Any other notable observations

Use only the supplied data and keep the response under 500 words.
"""
print(prompt[:500] + "...") # Print the beginning of the prompt to show its structure


This cell constructs a detailed prompt for the Gemini model, asking it to analyze the real estate `sample_csv` data for various insights.
*   `prompt = f"""..."""`: Defines a multi-line f-string to construct the prompt.
*   The prompt clearly specifies:
    *   That it needs to analyze the provided `sample_csv`.
    *   Specific aspects to cover: price trends, bedrooms/bathrooms, house size/acreage, geographic patterns, missing/unusual values, and other notable observations.
    *   Constraints: Use only the supplied data and keep the response under 500 words.
*   `print(prompt[:500] + "...")`: Prints the first 500 characters of the generated prompt to show its structure without overwhelming the output.

In [ ]:
# Send the prompt to the model for analysis
# Note: This will consume your API quota. Please ensure you have addressed any quota issues (e.g., TooManyRequests error) before running this cell.
try:
    response = model.generate_content(prompt)
    print("Analysis from Gemini model:")
    print(response.text)
except Exception as e:
    print(f"An error occurred while calling the model: {e}")

This cell sends the previously constructed analysis `prompt` to the Gemini model and prints the model's textual response.
*   `try...except`: This block handles potential errors during the API call.
*   `response = model.generate_content(prompt)`: Calls the `generate_content` method of the initialized `model` (which is `gemini-pro-latest`), passing the `prompt` for analysis. The model's analysis is stored in the `response` object.
*   `print("Analysis from Gemini model:")`: An informational message.
*   `print(response.text)`: Extracts and prints the text content (the analysis) from the `response` object.
*   `except Exception as e`: Catches any errors that occur during the model call and prints an error message.

In [ ]:
predict_prompt = f"""
Analyze the following real estate data provided in CSV format and estimate a predicted property price for each row.

Use the available property characteristics, including:
- status
- number of bedrooms
- number of bathrooms
- acreage
- city
- state
- ZIP code
- house size
- previous sold date, when available

Do not use the existing price column as an input when estimating the predicted price. Use it only afterward to compare the prediction with the actual listed price.

Return exactly these columns:
brokered_by,status,actual_price,predicted_price,bed,bath,acre_lot,city,state,zip_code,house_size

Requirements:
- Preserve the original row order.
- Include exactly one output row for every input row.
- Copy the original property information accurately.
- Put the original price in the actual_price column.
- Put your estimated value in the predicted_price column.
- Use numeric values only for actual_price and predicted_price.
- Do not include dollar signs or commas inside numeric values.
- Leave missing values blank.
- Return valid CSV data only.
- Do not include Markdown code fences.
- Do not include explanations, headings, or text before or after the CSV.

REAL ESTATE DATA:
{sample_csv}
"""

This cell constructs a specific prompt (`predict_prompt`) designed to instruct the Gemini model to perform price predictions on the `sample_csv` data.
*   `predict_prompt = f"""..."""`: Defines a multi-line f-string to construct the prediction prompt.
*   The prompt includes detailed instructions for the model:
    *   To analyze the real estate data and estimate a predicted property price for each row.
    *   Which property characteristics to use for estimation (status, bed, bath, acreage, city, state, ZIP code, house size, previous sold date).
    *   Explicitly states *not* to use the existing price column for prediction, but to use it later for comparison.
    *   Defines the exact output columns required in CSV format.
    *   Specifies output requirements: preserve order, one output row per input row, copy original info, put original price in `actual_price`, estimated price in `predicted_price`, use numeric values for prices (no symbols), leave missing values blank, return valid CSV only, and no markdown fences or extra text.
*   `REAL ESTATE DATA:\n{sample_csv}`: Injects the `sample_csv` data directly into the prompt for the model to process.

In [ ]:
try:
    prediction_response = model.generate_content(predict_prompt)

    print("Prediction output from Gemini:")
    print(prediction_response.text)

except Exception as e:
    print(f"An error occurred while generating predictions: {e}")

This cell sends the carefully crafted `predict_prompt` to the Gemini model to obtain property price predictions and then prints the model's raw CSV output.
*   `try...except`: This block handles potential errors during the API call.
*   `prediction_response = model.generate_content(predict_prompt)`: Calls the `generate_content` method of the `model` object, passing the `predict_prompt`. The model generates the predictions, and the result is stored in `prediction_response`.
*   `print("Prediction output from Gemini:")`: An informational message.
*   `print(prediction_response.text)`: Prints the raw text content of the `prediction_response`, which is expected to be a CSV string containing the actual and predicted prices.
*   `except Exception as e`: Catches any errors during the prediction generation and prints an error message.

In [ ]:
import pandas as pd
import io

# Assuming prediction_response.text contains only the CSV data as requested in the prompt
df_predicted = pd.read_csv(io.StringIO(prediction_response.text))
display(df_predicted.head())

This cell converts the CSV formatted prediction output received from the Gemini model into a pandas DataFrame for easier manipulation and analysis.
*   `import pandas as pd`: Imports the pandas library for data handling.
*   `import io`: Imports the `io` module to treat the string output as a file.
*   `df_predicted = pd.read_csv(io.StringIO(prediction_response.text))`: Reads the `prediction_response.text` (which is a CSV string) into a pandas DataFrame named `df_predicted`.
*   `display(df_predicted.head())`: Displays the first few rows of the `df_predicted` DataFrame, allowing you to quickly inspect its structure and content.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert price fields to numeric values
df_predicted['actual_price'] = pd.to_numeric(
    df_predicted['actual_price'],
    errors='coerce'
)

df_predicted['predicted_price'] = pd.to_numeric(
    df_predicted['predicted_price'],
    errors='coerce'
)

# Remove rows with missing price values
df_plot = df_predicted.dropna(
    subset=['actual_price', 'predicted_price']
).reset_index(drop=True)

# Plot actual and predicted property prices
plt.figure(figsize=(14, 6))

plt.plot(
    df_plot.index,
    df_plot['actual_price'],
    marker='o',
    label='Actual Property Price'
)

plt.plot(
    df_plot.index,
    df_plot['predicted_price'],
    marker='o',
    label='Predicted Property Price'
)

plt.title('Actual vs. Predicted Real Estate Prices')
plt.xlabel('Property Data Point')
plt.ylabel('Property Price ($)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

This cell visualizes the actual versus predicted property prices using a line plot to compare the model's predictions against the real values.
*   `import matplotlib.pyplot as plt`: Imports `matplotlib.pyplot` for creating plots.
*   `import pandas as pd`: Imports pandas for data manipulation (already imported but good practice for clarity in cell).
*   `df_predicted['actual_price'] = pd.to_numeric(...)`: Converts the 'actual_price' column to a numeric data type. `errors='coerce'` will turn any non-convertible values into `NaN` (Not a Number).
*   `df_predicted['predicted_price'] = pd.to_numeric(...)`: Converts the 'predicted_price' column to a numeric data type, handling potential errors.
*   `df_plot = df_predicted.dropna(subset=['actual_price', 'predicted_price']).reset_index(drop=True)`: Creates a new DataFrame `df_plot` by dropping rows where either 'actual_price' or 'predicted_price' are missing (`NaN`). `reset_index(drop=True)` re-indexes the DataFrame.
*   `plt.figure(figsize=(14, 6))`: Creates a new figure for the plot with a specified size.
*   `plt.plot(...)`: Plots the 'actual_price' and 'predicted_price' against their index (representing individual data points) as line plots with markers.
*   `plt.title(...)`, `plt.xlabel(...)`, `plt.ylabel(...)`: Sets the title and axis labels for the plot.
*   `plt.legend()`: Displays a legend to distinguish between actual and predicted prices.
*   `plt.grid(True)`: Adds a grid to the plot for better readability.
*   `plt.tight_layout()`: Adjusts plot parameters for a tight layout.
*   `plt.show()`: Displays the generated plot.

In [ ]:

# Make sure the price columns are numeric
df_predicted['actual_price'] = pd.to_numeric(
    df_predicted['actual_price'],
    errors='coerce'
)

df_predicted['predicted_price'] = pd.to_numeric(
    df_predicted['predicted_price'],
    errors='coerce'
)

# Remove rows where either price is missing
df_plot = df_predicted.dropna(
    subset=['actual_price', 'predicted_price', 'state']
).copy()

# Get the unique states represented in the prediction sample
states = df_plot['state'].unique()

# Determine the number of plots and grid dimensions
n_states = len(states)
n_cols = 3
n_rows = (n_states + n_cols - 1) // n_cols

# Create the subplot grid
fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(15, n_rows * 5)
)

# Convert axes to a one-dimensional array if it's an array
# This handles both (1,1) case (single Axes object) and (N,M) case (ndarray)
if isinstance(axes, np.ndarray):
    axes = axes.flatten()
else: # It's a single Axes object (e.g., from subplots(1,1))
    axes = [axes]

# Create one plot for each state
for i, state in enumerate(states):

    # Filter the dataframe for the current state
    state_data = df_plot[df_plot['state'] == state].reset_index(drop=True)

    # Plot actual and predicted property prices
    axes[i].plot(
        state_data.index,
        state_data['actual_price'],
        marker='o',
        label='Actual Property Price'
    )

    axes[i].plot(
        state_data.index,
        state_data['predicted_price'],
        marker='o',
        label='Predicted Property Price'
    )

    # Add titles and labels
    axes[i].set_title(f'Property Price Predictions for {state}')
    axes[i].set_xlabel('Property Data Point')
    axes[i].set_ylabel('Property Price ($)')
    axes[i].legend()
    axes[i].grid(True)

# Hide unused subplot spaces
for j in range(n_states, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

This cell generates separate plots of actual vs. predicted property prices for each unique state present in the data, allowing for state-specific performance analysis.
*   `df_predicted['actual_price'] = pd.to_numeric(...)`: Ensures the 'actual_price' column is numeric, coercing errors to `NaN`.
*   `df_predicted['predicted_price'] = pd.to_numeric(...)`: Ensures the 'predicted_price' column is numeric, coercing errors to `NaN`.
*   `df_plot = df_predicted.dropna(subset=['actual_price', 'predicted_price', 'state']).copy()`: Filters the DataFrame to remove rows with missing actual price, predicted price, or state, creating a copy to avoid `SettingWithCopyWarning`.
*   `states = df_plot['state'].unique()`: Extracts a list of all unique states from the filtered data.
*   `n_states = len(states)`: Counts the number of unique states.
*   `n_cols = 3`: Sets the number of columns for the subplot grid.
*   `n_rows = (n_states + n_cols - 1) // n_cols`: Calculates the number of rows needed for the subplot grid based on the number of states and columns.
*   `fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))`: Creates a figure and a grid of subplots (`axes`) with the calculated rows and columns.
*   `if isinstance(axes, np.ndarray): ... else: ...`: This block ensures `axes` is a flat, one-dimensional array or list of subplot objects, regardless of how `plt.subplots` structured its return (single Axes object or NumPy array of Axes).
*   `for i, state in enumerate(states):`: Loops through each unique state.
    *   `state_data = df_plot[df_plot['state'] == state].reset_index(drop=True)`: Filters the data for the current state and resets its index.
    *   `axes[i].plot(...)`: Plots the actual and predicted prices for the current `state_data` on its dedicated subplot.
    *   `axes[i].set_title(...)`, `axes[i].set_xlabel(...)`, `axes[i].set_ylabel(...)`, `axes[i].legend()`, `axes[i].grid(True)`: Configures the title, labels, legend, and grid for the current state's subplot.
*   `for j in range(n_states, len(axes)): fig.delaxes(axes[j])`: Hides any empty subplots if the number of states doesn't perfectly fill the grid.
*   `plt.tight_layout()`: Adjusts subplot parameters for a tight layout.
*   `plt.show()`: Displays all the generated plots.